# Step 7 · Resolve modular feature profiles

V4 assembles a profile from reusable modules in a defined order: core, pathogen group, host group, host taxon, and pair context. Resolution records both applicable features and reasons for features that do not apply.

This phase resolves configuration only. Scoring decisions are kept in the decision ledger and use their own scope-specific cache keys.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository')
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id, log_step
RUN_ID = configured_run_id(PROJECT_ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('07_resolve_feature_profiles', RUN_ID)
print('Project root:', ROOT, '| log:', LOG_PATH)

2026-11-03 00:46 INFO Notebook session started at 2026-11-03T00:46:09.077234+00:00


2026-11-03 00:46 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4


2026-11-03 00:46 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/07_resolve_feature_profiles.log


Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4 | log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/07_resolve_feature_profiles.log


## Step 1 · Load and review the feature catalogue

In [2]:
import yaml, pandas as pd
from IPython.display import display
from pathogen_atlas_v4.features import resolve_profile
CATALOGUE_PATH = ROOT / 'configs/features.yaml'
CATALOGUE = yaml.safe_load(CATALOGUE_PATH.read_text())
print('Catalogue:', CATALOGUE_PATH)
print('Catalogue version:', CATALOGUE.get('catalogue_version'))
display(pd.DataFrame([{'module': key, 'feature_count': len(features)} for key, features in CATALOGUE['modules'].items()]))
LOG.info('Feature catalogue modules=%d', len(CATALOGUE['modules']))

Catalogue: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/configs/features.yaml
Catalogue version: v4-feature-catalogue-1.0.2


,module,feature_count
0,core,1
1,pathogen_group:bacteria,5
2,pathogen_group:fungi,4
3,pathogen_group:oomycetes,3
4,pathogen_group:parasites,4
5,pathogen_group:viruses,3
6,host_group:human,100
7,host_group:livestock,0
8,host_group:crop,0
9,pair_context:disease,43


2026-11-03 00:46 INFO Feature catalogue modules=11


## Step 2 · Resolve one profile for each pilot host context

In [3]:
import yaml, json
from pathogen_atlas_v4.runs import resolve_run
CONFIG = yaml.safe_load((ROOT / 'configs/pilot_v1.yaml').read_text())
STAGING_TAXONOMY_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/taxonomy'))[-1]
RUN_TAXONOMY_ROOT = ROOT / 'data/v4/runs' / CONFIG['run']['run_id'] / 'taxonomy'
if not (RUN_TAXONOMY_ROOT / 'pathogens.parquet').exists():
    raise FileNotFoundError('Run taxonomy was not resolved; run 00_configure_run.ipynb first.')
hosts = pd.read_parquet(STAGING_TAXONOMY_ROOT / 'hosts.parquet')
run_pathogens = pd.read_parquet(RUN_TAXONOMY_ROOT / 'pathogens.parquet')
resolved = resolve_run(CONFIG, known_host_ids=set(hosts.host_id.astype(str)),
    known_pathogen_ids=set(run_pathogens.pathogen_id.astype(str)),
    host_group_ids={k:g.host_id.astype(str).tolist() for k,g in hosts.groupby('host_group')},
    pathogen_group_ids={k:g.pathogen_id.astype(str).tolist() for k,g in run_pathogens.groupby('pathogen_group')})
selected_host_ids = set(resolved.host_ids)
PATHOGEN_GROUPS = sorted(run_pathogens.loc[run_pathogens.pathogen_id.astype(str).isin(resolved.pathogen_ids), 'pathogen_group'].astype(str).unique())
hosts = hosts.loc[hosts.host_id.astype(str).isin(selected_host_ids)].copy()
PROFILE_VERSION = str(CONFIG.get('features', {}).get('profile_version', CATALOGUE.get('profile_version', '')))
profiles = []
for host in hosts.itertuples(index=False):
    for pathogen_group in PATHOGEN_GROUPS:
        profile = resolve_profile(CATALOGUE, pathogen_group=pathogen_group,
            host_group=host.host_group, host_taxon_id=host.taxon_id,
            relationship_context='disease', profile_version=PROFILE_VERSION)
        profiles.append({'host_id': host.host_id, 'host': host.accepted_name,
                         'host_group': host.host_group, 'pathogen_group': pathogen_group,
                         'resolved_profile': profile})
if not profiles:
    raise RuntimeError('No feature profiles match the selected run hosts and pathogen groups.')
profile_summary = pd.DataFrame([{'host_id': p['host_id'], 'host': p['host'],
    'host_group': p['host_group'], 'pathogen_group': p['pathogen_group'],
    'applicable_features': len(p['resolved_profile']['ordered_features']),
    'not_applicable_features': len(p['resolved_profile']['not_applicable']),
    'profile_hash': p['resolved_profile']['resolved_profile_hash']} for p in profiles])
display(profile_summary)
LOG.info('Resolved %d host × pathogen-group profiles', len(profiles))

,host_id,host,host_group,pathogen_group,applicable_features,not_applicable_features,profile_hash
0,ncbi:4530,Oryza sativa,crop,viruses,40,135,229863f419db9c27ad4c866727791920e61f0042be3b22...


2026-11-03 00:46 INFO Resolved 1 host × pathogen-group profiles


## Step 3 · Inspect feature scopes and exclusions

In [4]:
selected_profile = profiles[0]['resolved_profile']  # example from this run's selected host/group scope
selected_features = pd.DataFrame(selected_profile['ordered_features'])
not_applicable = pd.DataFrame(selected_profile['not_applicable'])
print('Example profile:', selected_profile['profile_id'], selected_profile['profile_version'])
print('Included feature scopes:')
display(selected_features.groupby(['module','scope']).size().rename('features').to_frame())
print('Not-applicable reasons:')
display(not_applicable.head(25))
LOG.info('Example resolved profile hash=%s included=%d excluded=%d', selected_profile['resolved_profile_hash'], len(selected_features), len(not_applicable))

Example profile: v4-host-aware-feature-questions v4-feature-profile-1.0.1
Included feature scopes:


,,features
module,scope,
core,pathogen_host_pair,1
host_taxon:4530,pathogen_host_pair,12
pair_context:disease,pathogen_host_pair,24
pathogen_group:viruses,pathogen,3


Not-applicable reasons:


,feature_id,reason,module,selector,required_value
0,animal_respiratory_disease,requires host traits: host:animal,pair_context:disease,NaN,NaN
1,animal_digestive_disease,requires host traits: host:animal,pair_context:disease,NaN,NaN
2,animal_neurologic_disease,requires host traits: host:animal,pair_context:disease,NaN,NaN
3,animal_reproductive_disease,requires host traits: host:animal,pair_context:disease,NaN,NaN
4,animal_mortality,requires host traits: host:animal,pair_context:disease,NaN,NaN
5,animal_morbidity,requires host traits: host:animal,pair_context:disease,NaN,NaN
6,animal_shedding,requires host traits: host:animal,pair_context:disease,NaN,NaN
7,animal_vertical_transmission,requires host traits: host:animal,pair_context:disease,NaN,NaN
8,animal_vector_transmission,requires host traits: host:animal,pair_context:disease,NaN,NaN
9,animal_zoonotic_spillover,requires host traits: host:animal,pair_context:disease,NaN,NaN


2026-11-03 00:46 INFO Example resolved profile hash=229863f419db9c27ad4c866727791920e61f0042be3b228170dd96ea21b13671 included=40 excluded=135


## Step 4 · Save resolved profiles for the run

In [5]:
import json
PROFILE_DIR = ROOT / 'data/v4/runs' / RUN_ID / 'feature_profiles'
PROFILE_DIR.mkdir(parents=True, exist_ok=True)
# The run selection may have changed; do not leave profiles from a prior host cohort.
for stale_profile in PROFILE_DIR.glob('*.json'):
    stale_profile.unlink()
PROFILE_PATHS = []
for item in profiles:
    profile = item['resolved_profile']
    path = PROFILE_DIR / f'{profile["resolved_profile_hash"]}.json'
    path.write_text(json.dumps(profile, indent=2, ensure_ascii=False) + '\n')
    PROFILE_PATHS.append(path)
print('Saved profiles:', len(PROFILE_PATHS), '| directory:', PROFILE_DIR)
LOG.info('Saved %d selected-run feature profiles under %s', len(PROFILE_PATHS), PROFILE_DIR)

2026-11-03 00:46 INFO Saved 1 selected-run feature profiles under /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/pilot-v1/feature_profiles


Saved profiles: 1 | directory: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/runs/pilot-v1/feature_profiles


## Step 5 · Scoring handoff

In [6]:
print('Profile resolution is local and deterministic; no model calls were made.')
print('The scoring stage should generate pathogen-, host-, or pair-scoped tasks from these ordered feature definitions.')
LOG.info('Feature scoring handoff ready; model calls=0')

2026-11-03 00:46 INFO Feature scoring handoff ready; model calls=0


Profile resolution is local and deterministic; no model calls were made.
The scoring stage should generate pathogen-, host-, or pair-scoped tasks from these ordered feature definitions.
